In [5]:
import pandas as pd
import numpy as np
df = pd.read_excel('/content/Employee_data (2).xlsx')
print("--- ORIGINAL DATAFRAME ---")
print(df)
print("\n")


--- ORIGINAL DATAFRAME ---
    Employee_ID              Name   Department    Age    Salary  Join_Date
0           101          John Doe  Engineering   28.0   75000.0 2021-01-15
1           102        Jane Smith    Marketing   34.0   82000.0 2019-03-12
2           103      Mike Johnson  Engineering    NaN   90000.0 2020-06-01
3           104    Sarah Williams           HR   29.0   62000.0 2022-02-20
4           105      Robert Brown        Sales   45.0   55000.0 2018-11-10
5           101          John Doe  Engineering   28.0   75000.0 2021-01-15
6           106       Emily Davis    Marketing   31.0       NaN 2021-07-22
7           107      Chris Miller        Sales   22.0   48000.0 2023-01-05
8           108       Anna Taylor           HR  205.0   72000.0 2020-10-15
9           109      David Wilson  Engineering   40.0  120000.0 2015-05-19
10          110       Linda Moore        Sales   37.0   64000.0 2017-08-30
11          111    James Anderson          ???   29.0   59000.0 2022-04-1

In [6]:
# Check for missing values
print(f"Missing values per column:\n{df.isnull().sum()}\n")

Missing values per column:
Employee_ID    0
Name           0
Department     0
Age            2
Salary         2
Join_Date      0
dtype: int64



In [7]:
# Method 1: Drop rows with missing values (Destructive)
df_dropped = df.dropna()
print("1. Shape after dropping rows with NaNs:", df_dropped.shape)

print(df_dropped)


1. Shape after dropping rows with NaNs: (46, 6)
    Employee_ID              Name   Department    Age    Salary  Join_Date
0           101          John Doe  Engineering   28.0   75000.0 2021-01-15
1           102        Jane Smith    Marketing   34.0   82000.0 2019-03-12
3           104    Sarah Williams           HR   29.0   62000.0 2022-02-20
4           105      Robert Brown        Sales   45.0   55000.0 2018-11-10
5           101          John Doe  Engineering   28.0   75000.0 2021-01-15
7           107      Chris Miller        Sales   22.0   48000.0 2023-01-05
8           108       Anna Taylor           HR  205.0   72000.0 2020-10-15
9           109      David Wilson  Engineering   40.0  120000.0 2015-05-19
10          110       Linda Moore        Sales   37.0   64000.0 2017-08-30
11          111    James Anderson          ???   29.0   59000.0 2022-04-12
12          112    Barbara Thomas    Marketing   33.0   78000.0 2019-09-01
13          113   Richard Jackson        Sales   42.

In [8]:
# Method 2: Imputation (Filling with Mean/Median/Mode)
# We will use Median for Age to fill NaNs (robust to outliers)
df_imputed = df.copy()
median_age = df_imputed['Age'].median()
df_imputed['Age'] = df_imputed['Age'].fillna(median_age)

print(f"2. Filled missing Age with median ({median_age}):")
print(df_imputed)
print("\n")

2. Filled missing Age with median (32.0):
    Employee_ID              Name   Department    Age    Salary  Join_Date
0           101          John Doe  Engineering   28.0   75000.0 2021-01-15
1           102        Jane Smith    Marketing   34.0   82000.0 2019-03-12
2           103      Mike Johnson  Engineering   32.0   90000.0 2020-06-01
3           104    Sarah Williams           HR   29.0   62000.0 2022-02-20
4           105      Robert Brown        Sales   45.0   55000.0 2018-11-10
5           101          John Doe  Engineering   28.0   75000.0 2021-01-15
6           106       Emily Davis    Marketing   31.0       NaN 2021-07-22
7           107      Chris Miller        Sales   22.0   48000.0 2023-01-05
8           108       Anna Taylor           HR  205.0   72000.0 2020-10-15
9           109      David Wilson  Engineering   40.0  120000.0 2015-05-19
10          110       Linda Moore        Sales   37.0   64000.0 2017-08-30
11          111    James Anderson          ???   29.0   59

In [9]:
# We will use the IQR (Interquartile Range) method to remove Age outliers (e.g., 120).

Q1 = df_imputed['Age'].quantile(0.25)
Q3 = df_imputed['Age'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Age Bounds: {lower_bound} to {upper_bound}")

# Filter the data
df_clean_noise = df_imputed[
    (df_imputed['Age'] >= lower_bound) &
    (df_imputed['Age'] <= upper_bound)
]

print("Rows removed (Noise):")
print(df_imputed[~df_imputed.index.isin(df_clean_noise.index)])
print("\n")
print("Dataset after noise removal")
print(df_clean_noise)

Age Bounds: 15.875 to 50.875
Rows removed (Noise):
    Employee_ID            Name Department    Age   Salary  Join_Date
8           108     Anna Taylor         HR  205.0  72000.0 2020-10-15
20          119  Nancy Martinez      Sales   -5.0  63000.0 2018-04-10


Dataset after noise removal
    Employee_ID              Name   Department   Age    Salary  Join_Date
0           101          John Doe  Engineering  28.0   75000.0 2021-01-15
1           102        Jane Smith    Marketing  34.0   82000.0 2019-03-12
2           103      Mike Johnson  Engineering  32.0   90000.0 2020-06-01
3           104    Sarah Williams           HR  29.0   62000.0 2022-02-20
4           105      Robert Brown        Sales  45.0   55000.0 2018-11-10
5           101          John Doe  Engineering  28.0   75000.0 2021-01-15
6           106       Emily Davis    Marketing  31.0       NaN 2021-07-22
7           107      Chris Miller        Sales  22.0   48000.0 2023-01-05
9           109      David Wilson  Engineer

In [10]:
duplicates = df_clean_noise[df_clean_noise.duplicated(keep=False)]
print("Duplicate Rows found:")
print(duplicates)

# Remove duplicates (keep the first occurrence)
df_final = df_clean_noise.drop_duplicates(keep='first')

print("\n--- FINAL CLEANED DATAFRAME ---")
print(df_final)


Duplicate Rows found:
    Employee_ID            Name   Department   Age   Salary  Join_Date
0           101        John Doe  Engineering  28.0  75000.0 2021-01-15
5           101        John Doe  Engineering  28.0  75000.0 2021-01-15
12          112  Barbara Thomas    Marketing  33.0  78000.0 2019-09-01
16          112  Barbara Thomas    Marketing  33.0  78000.0 2019-09-01
29          128      Alice King    Marketing  31.0  80000.0 2019-05-05
30          128      Alice King    Marketing  31.0  80000.0 2019-05-05

--- FINAL CLEANED DATAFRAME ---
    Employee_ID              Name   Department   Age    Salary  Join_Date
0           101          John Doe  Engineering  28.0   75000.0 2021-01-15
1           102        Jane Smith    Marketing  34.0   82000.0 2019-03-12
2           103      Mike Johnson  Engineering  32.0   90000.0 2020-06-01
3           104    Sarah Williams           HR  29.0   62000.0 2022-02-20
4           105      Robert Brown        Sales  45.0   55000.0 2018-11-10
6   